# 03. Conditions, loops, functions, lambda, and OOP

**Level:** Beginner

**Prerequisites:** 02: containers.

[Course index](00_START_HERE.ipynb) | [Previous](02_Python_Variables_Strings_and_Containers.ipynb) | [Next](04_Files_JSON_and_Data_Validation.ipynb)

**Topics:** Conditions, Loops, Functions, lambda, Classes, OOP, Exceptions.

**Matching script:** `examples\03_Python_Control_Functions_and_OOP.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

An if statement selects behavior, and a for loop processes an iterable. A comprehension is
useful for a short transformation or filter; use an ordinary loop when error handling or
multiple state changes would make it hard to read.

A function groups a reusable operation. Parameters are input names, and return sends a result
to the caller. lambda creates a short anonymous function, often a sort key, not a replacement
for a full workflow. Default and keyword-only arguments make policies explicit at call sites.

A class combines related state and behavior. __init__ initializes an instance; self denotes
that instance by convention. A policy object can hold a threshold, while a function can do a
stateless calculation. Prefer the smallest structure that expresses ownership and invariants.

Exceptions signal invalid input or failure. Raise TypeError for an unsupported type and
ValueError for an unacceptable value of a supported type. Catch only a known expected exception.
Do not turn every failure into a zero-valued prediction. Assertions in the examples verify
behavior; production input validation uses explicit raises because assertions can be disabled.

## Mathematics or data contract

A threshold classifier predicts 1 when value >= threshold; equality at the boundary must be documented.

## Important limitation

This threshold is a hand-written rule, not a learned classifier.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Reusable rule object with explicit validation

**Purpose:** Follow an input through a method, branch, returned label, and rejection path.

### Read the code in this order
1. Store a threshold on one instance.
2. Validate input before comparing.
3. Use a function and loop to score records.
4. Sort with lambda and prove invalid input raises without a fake result.

### Syntax and behavior to notice
- `predict(...)` returns the estimator's chosen output, not a guarantee of correctness.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `raise` reports invalid input or a failed requirement instead of fabricating success.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
latency cannot be negative
sorted: [{'ms': 5}, {'ms': 20}] labels: [0, 1, 1]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
class LatencyRule:
    def __init__(self, threshold):
        self.threshold = threshold

    def predict(self, latency_ms):
        if not isinstance(latency_ms, (int, float)) or isinstance(latency_ms, bool):
            raise TypeError("numeric latency required")
        if latency_ms < 0:
            raise ValueError("latency cannot be negative")
        return int(latency_ms >= self.threshold)

def score(values, rule):
    return [rule.predict(value) for value in values]

rule = LatencyRule(20)
assert score([5, 20, 30], rule) == [0, 1, 1]
ranked = sorted([{"ms": 20}, {"ms": 5}], key=lambda row: row["ms"])
try:
    rule.predict(-1)
except ValueError as error:
    print(str(error))
else:
    raise AssertionError("invalid input accepted")
print("sorted:", ranked, "labels:", score([5, 20, 30], rule))

latency cannot be negative
sorted: [{'ms': 5}, {'ms': 20}] labels: [0, 1, 1]


### Interpret example 1

The result should demonstrate this purpose: **Follow an input through a method, branch, returned label, and rejection path.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Compose a predictor and a reporting service

**Purpose:** Use dependency injection rather than a large inheritance hierarchy.

### Read the code in this order
1. Define a small public prediction contract.
2. Inject a fake into a service.
3. Verify the same service can use a different compatible collaborator.

### Syntax and behavior to notice
- `predict(...)` returns the estimator's chosen output, not a guarantee of correctness.
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
same service, independently injected policies
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
class ConstantPredictor:
    def __init__(self, label):
        self.label = label
    def predict(self, features):
        return self.label

class ReportService:
    def __init__(self, predictor):
        self.predictor = predictor
    def report(self, features):
        return {"prediction": self.predictor.predict(features)}

normal = ReportService(ConstantPredictor("normal"))
alert = ReportService(ConstantPredictor("alert"))
assert normal.report([10, 0]) == {"prediction": "normal"}
assert alert.report([10, 0]) == {"prediction": "alert"}
print("same service, independently injected policies")

same service, independently injected policies


### Interpret example 2

The result should demonstrate this purpose: **Use dependency injection rather than a large inheritance hierarchy.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add a second keyword-only threshold for packet loss and test an exact boundary and invalid input.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Conditions` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A threshold classifier predicts 1 when value >= threshold; equality at the boundary must be documented.
- Name the boundary of the demonstration: This threshold is a hand-written rule, not a learned classifier.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://docs.python.org/3.13/tutorial/`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.